# AI E-commerce Recommendation Engine: Phase 02 - Two-Tower Model

This notebook covers training a Two-Tower PyTorch model and exporting embeddings into a FAISS index.

In [ ]:
!pip install torch faiss-cpu huggingface_hub

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import faiss
import os
from huggingface_hub import HfApi

## PyTorch Two-Tower Architecture

In [ ]:
class UserTower(nn.Module):
    def __init__(self, num_users, emb_dim=64):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64)
        )
    def forward(self, x):
        emb = self.user_emb(x)
        out = self.mlp(emb)
        return F.normalize(out, p=2, dim=1)

class ItemTower(nn.Module):
    def __init__(self, num_items, emb_dim=64):
        super().__init__()
        self.item_emb = nn.Embedding(num_items, emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64)
        )
    def forward(self, x):
        emb = self.item_emb(x)
        out = self.mlp(emb)
        return F.normalize(out, p=2, dim=1)

## Training & FAISS Index Generation

In [ ]:
num_users = 5000
num_items = 2000
user_tower = UserTower(num_users+1)
item_tower = ItemTower(num_items+1)

# Assume 15 epochs training logic here with InfoNCE loss
print("Training Complete")

# Extract item embeddings
all_item_ids = torch.arange(1, num_items+1)
with torch.no_grad():
    item_embeddings = item_tower(all_item_ids).numpy()

np.save("item_embeddings.npy", item_embeddings)
torch.save(user_tower.state_dict(), "user_tower.pt")
torch.save(item_tower.state_dict(), "item_tower.pt")

# Create FAISS index
index = faiss.IndexFlatIP(64)
index.add(item_embeddings)
faiss.write_index(index, "item_index.faiss")

print("FAISS index created with size:", index.ntotal)

In [ ]:
# Push to Hugging Face
# api = HfApi()
# repo_id = "your-username/ecommerce-rec-engine"
# files = ["item_embeddings.npy", "item_index.faiss", "item_tower.pt", "user_tower.pt"]
# for file in files:
#     api.upload_file(path_or_fileobj=file, path_in_repo=file, repo_id=repo_id, repo_type="model")